In [6]:
import os

# os.environ["CUDA_VISIBLE_DEVICES"] = "0"  
os.environ["CUDA_VISIBLE_DEVICES"] = "2,3"  
from torch import nn


import numpy as np                                                                  
import torch                                          

                              
from transformers import AutoModelForCausalLM, AutoTokenizer  
from tqdm import tqdm
import matplotlib.pyplot as plt  
import torch.nn.functional as F
import gc
import re
import copy

import sys
sys.path.append('..')
import JCBScope_utils

# Move to GPU with optimal dtype
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# device = "cpu"


In [7]:
# Load the tokenizer and model

model_name = "meta-llama/Llama-3.2-1B"
# model_name = "meta-llama/Llama-3.2-3B"
# model_name = "meta-llama/Llama-3.1-8B"
model_name_short = model_name.split("/")[-1]
if device == "cpu":
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)
    model = model.to(device)
else:
    tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="auto")
    model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")
    
embedding_layer = model.get_input_embeddings()
embed_device = embedding_layer.weight.device    

In [8]:
front_pad = 0
back_pad = 0
front_strip = 0

# num_prompts = 100
num_prompts = 300

# Get special tokens if available
bos_token_id = tokenizer.bos_token_id if tokenizer.bos_token_id is not None else tokenizer.cls_token_id
eos_token_id = tokenizer.eos_token_id if tokenizer.eos_token_id is not None else tokenizer.sep_token_id


In [9]:
def compute_kl_divergence(log_probs_p, log_probs_q, eps=1e-10):
    """KL(P || Q) = sum P * (log P - log Q). Inputs are log-probabilities."""
    p = torch.exp(log_probs_p).clamp(min=eps)
    q = torch.exp(log_probs_q).clamp(min=eps)
    return (p * (log_probs_p - torch.log(q))).sum().item()


def loo_kl_ranking(string, verbose=True):
    """
    Leave-One-Out (LOO) ranking: for each token, zero it out and record the change in
    KL divergence of the predicted token distribution. Returns ranked token indices,
    decoded tokens, and corresponding KL divergences.
    """
    input_ids_list = tokenizer(string, add_special_tokens=False)["input_ids"]
    if eos_token_id is not None:
        input_ids_list += [eos_token_id] * back_pad

    input_ids = torch.tensor([input_ids_list], dtype=torch.long).to(embed_device)
    attention_mask = torch.ones_like(input_ids, device=embed_device)
    seq_len = input_ids.size(1)

    decoded_tokens = tokenizer.batch_decode([[tid] for tid in input_ids[0].tolist()], skip_special_tokens=True)

    
    token_idx = [idx for idx in range(front_pad, len(decoded_tokens), 1)][front_strip:]
    decoded_tokens_list = [decoded_tokens[idx] for idx in token_idx]

    d_model = embedding_layer.embedding_dim
    # Use plain tensor (no gradient needed for LOO)
    residual = torch.zeros(len(token_idx), d_model, device=embed_device)
    presence = torch.ones(len(decoded_tokens), 1, device=embed_device)

    loss_position = seq_len - 2

    model.eval()
    forward_pass = JCBScope_utils.customize_forward_pass(
        model, residual, presence, input_ids, token_idx, attention_mask
    )

    # Full-context forward pass (no ablation) - no gradient needed
    with torch.no_grad():
        _, logits_orig = forward_pass(
            loss_position=loss_position,
            hidden_norm_as_loss=False,
            unnormalized_logits=False,
            tie_input_output_embed=False,
        )
    log_probs_orig = F.log_softmax(logits_orig[loss_position].float(), dim=-1)
    del logits_orig
    gc.collect()
    if device != "cpu":
        torch.cuda.empty_cache()

    if verbose:
        print(f"Computing LOO KL ranking ({len(token_idx)} tokens)...")

    # Inner loop: zero out each token one at a time, compute KL divergence
    kl_values = []
    for i, idx in enumerate(tqdm(token_idx, desc="LOO tokens", leave=False, disable=not verbose)):
        presence_loo = presence.clone()
        presence_loo[idx, 0] = 0.0

        forward_pass_loo = JCBScope_utils.customize_forward_pass(
            model, residual, presence_loo, input_ids, token_idx, attention_mask
        )
        with torch.no_grad():
            _, logits_loo = forward_pass_loo(
                loss_position=loss_position,
                hidden_norm_as_loss=False,
                unnormalized_logits=False,
                tie_input_output_embed=False,
            )
        log_probs_loo = F.log_softmax(logits_loo[loss_position].float(), dim=-1)
        kl = compute_kl_divergence(log_probs_orig.cpu(), log_probs_loo.cpu())
        kl_values.append(kl)
        del logits_loo, forward_pass_loo
        if device != "cpu":
            torch.cuda.empty_cache()

    del forward_pass

    # Rank by KL divergence (descending: higher KL = more important token)
    kl_values = np.array(kl_values)
    rank_order = np.argsort(kl_values)[::-1]

    ranked_token_indices = [int(token_idx[j]) for j in rank_order]
    ranked_decoded_tokens = [decoded_tokens_list[j] for j in rank_order]
    ranked_kl_divergences = [float(kl_values[j]) for j in rank_order]

    true_token_id = input_ids[0, loss_position + 1].item()
    predicted_token_id = log_probs_orig.argmax().item()
    true_token_str = tokenizer.decode([true_token_id])
    predicted_token_str = tokenizer.decode([predicted_token_id])

    if verbose:
        print(string)
        print(f"True: {true_token_str!r} | Predicted: {predicted_token_str!r}")
        print(f"Top-5 LOO tokens (by KL): {ranked_decoded_tokens[:5]}")

    return {
        "ranked_token_indices": ranked_token_indices,
        "decoded_tokens": ranked_decoded_tokens,
        "kl_divergences": ranked_kl_divergences,
        "true_token": true_token_str,
        "predicted_token": predicted_token_str,
    }

In [10]:
import json
from pathlib import Path

# Load prompts from JSON
prompts_path = Path("../data/lambada_prompts.json")
with open(prompts_path, "r", encoding="utf-8") as f:
    all_prompts_data = json.load(f)

# Handle both list-of-dicts and {prompts: [...]} formats
if isinstance(all_prompts_data, dict) and "prompts" in all_prompts_data:
    prompts_list = all_prompts_data["prompts"]
else:
    prompts_list = all_prompts_data

prompts_to_process = prompts_list[:num_prompts]

# Label for LOO results
label = f"{model_name_short}__LOO_KL_lambada"
results = []
for i, item in enumerate(tqdm(prompts_to_process, desc="Processing prompts")):
    print(f"processing {i+1} of {len(prompts_to_process)} prompts")
    
    prompt = item["text"] if isinstance(item, dict) else item
    # print(prompt)
    loo_result = loo_kl_ranking(string=prompt, verbose=True)
    # Keep last successful result for visualization
    _last_loo = loo_result
    entry = {
        "prompt": prompt,
        "index": i,
        "ranked_token_indices": loo_result["ranked_token_indices"],
        "decoded_tokens": loo_result["decoded_tokens"],
        "kl_divergences": loo_result["kl_divergences"],
        "true_token": loo_result["true_token"],
        "predicted_token": loo_result["predicted_token"],
    }
    if isinstance(item, dict):
        entry.update({k: v for k, v in item.items() if k != "text" and k != "prompt"})
    results.append(entry)




Processing prompts:   0%|          | 0/300 [00:00<?, ?it/s]

processing 1 of 300 prompts
Computing LOO KL ranking (69 tokens)...


Processing prompts:   0%|          | 1/300 [00:13<1:07:46, 13.60s/it]

She had never been inside his house before. It was small and surprisingly neat for a man who lived alone. The furniture was sparse but of good quality. A number of photographs sat on the mantel. She moved closer for a better look. The first depicted a young couple holding a little boy. There were three other photos of the same couple
True: ' couple' | Predicted: ' couple'
Top-5 LOO tokens (by KL): [' same', ' of', ' the', ' couple', ' other']
processing 2 of 300 prompts
Computing LOO KL ranking (105 tokens)...


Processing prompts:   1%|          | 2/300 [00:35<1:30:55, 18.31s/it]

...

More light carriers were in various stages of completion, and the first heavy carrier, massing half a million tons, was just starting its two year construction phase. Defiant was built to carry a total of 34 smaller craft. While she was streamlined and capable of skimming gas giants herself, she would also carry four Mark 4 fuel shuttles, five personnel shuttles and 25 fighters. Her crew would total almost 500. Shiloh had already studied her specs, and he was impressed
True: ' impressed' | Predicted: ' impressed'
Top-5 LOO tokens (by KL): [' was', 'More', ' studied', ' and', ' he']
processing 299 of 300 prompts
Computing LOO KL ranking (85 tokens)...


Processing prompts: 100%|█████████▉| 299/300 [1:23:00<00:18, 18.28s/it]

Meghan panicked for a brief second, but it was long enough for her to make the mistake of trying to stop. She braked and swerved, sending her car into a spin. The world whirled around her in a blindingly white display before the side of her car slammed into something hard. Her head hit the window and everything winked out of existence.

Alexander rushed over the snow to check on Meghan
True: ' Meghan' | Predicted: ' her'
Top-5 LOO tokens (by KL): [' on', ' check', 'Meg', 'han', ' rushed']
processing 300 of 300 prompts
Computing LOO KL ranking (82 tokens)...


Processing prompts: 100%|██████████| 300/300 [1:23:17<00:00, 16.66s/it]

In the theater lobby, she once again purchased a small bag of popcorn and a cold soda, then walked into the dimly lit theater.

She saw that Mark was already in one of the wheelchair spaces. Tessa had been right; this was the perfect movie.

Without hesitation, Barbie moved around the back and entered the row from the opposite direction. She sat down, leaving one empty seat between her and Mark
True: ' Mark' | Predicted: ' Mark'
Top-5 LOO tokens (by KL): ['In', ' and', ' Mark', ' between', '.']


In [11]:
label

'Llama-3.2-1B__LOO_KL_lambada'

In [12]:
# Save LOO results to JSON
result_path = Path(f"../results/{label}_loo_results.json")
result_path.parent.mkdir(parents=True, exist_ok=True)
with open(result_path, "w", encoding="utf-8") as f:
    json.dump({"label": label, "results": results}, f, indent=2, ensure_ascii=False)

print(f"Saved {len(results)} LOO results to {result_path}")

# Restore last successful result for visualization cells
if "_last_loo" in dir():
    _last_ranked_indices = _last_loo["ranked_token_indices"]
    _last_decoded_tokens = _last_loo["decoded_tokens"]
    _last_kl_divergences = _last_loo["kl_divergences"]

Saved 300 LOO results to ../results/Llama-3.2-1B__LOO_KL_lambada_loo_results.json
